<a href="https://colab.research.google.com/github/mvashi-sonic/AICapstoneProj/blob/dev/capstone_FAISS_retriever.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
pip install sentence-transformers faiss-cpu

In [2]:
from google.colab import drive

# 1. Mount your Google Drive
drive.mount('/content/drive')

Mounted at /content/drive


In [3]:
import json
import pickle
import numpy as np
import faiss

from sentence_transformers import SentenceTransformer

In [4]:
embedder = SentenceTransformer(
    "sentence-transformers/all-MiniLM-L6-v2"
)

modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/10.5k [00:00<?, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/612 [00:00<?, ?B/s]

...

tokenizer_config.json:   0%|          | 0.00/350 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

In [5]:
print(embedder.get_sentence_embedding_dimension())

384


/tmp/ipykernel_533/1832433109.py:1: FutureWarning: The `get_sentence_embedding_dimension` method has been renamed to `get_embedding_dimension`.
  print(embedder.get_sentence_embedding_dimension())


In [26]:
chunks = []

with open("/content/drive/MyDrive/capstone_usable_qa_data/all_records.json", encoding="utf-8") as f:
    records = json.load(f)
    for record in records["All_Records"]:
        chunks.append(record)


Further changes in the tax laws of foreign jurisdictions could arise as a result of the base erosion and profit shifting (BEPS) project that was undertaken by the Organization for Economic Co-operation and Development (OECD)
We also engage in acquisitions and other transactions to meet certain technology needs, to obtain development resources or open or expand opportunities for our technologies and to support the design and introduction of new products and services (or enhance existing products and services)
Inventories We charge cost of sales for inventory provisions to write-down our inventory to the lower of cost or net realizable value or for obsolete or excess inventory, and for excess product purchase commitments. Most of our inventory provisions relate to excess quantities of products or componen


In [33]:
paragraphs = [
    chunk["metadata"] + chunk["reference"]
    for chunk in chunks
]
print(paragraphs[0][:300])
print(paragraphs[1][:300])
print(paragraphs[2][:300])

Reference Ticker-QCOM CompanyName-Qualcomm Incorporated Date-2023 Section-1AFurther changes in the tax laws of foreign jurisdictions could arise as a result of the base erosion and profit shifting (BEPS) project that was undertaken by the Organization for Economic Co-operation and Development (OECD)
Reference Ticker-NVDA CompanyName-NVIDIA Corporation Date-2025 Section-1We also engage in acquisitions and other transactions to meet certain technology needs, to obtain development resources or open or expand opportunities for our technologies and to support the design and introduction of new produ
Reference Ticker-NVDA CompanyName-NVIDIA Corporation Date-2025 Section-7Inventories We charge cost of sales for inventory provisions to write-down our inventory to the lower of cost or net realizable value or for obsolete or excess inventory, and for excess product purchase commitments. Most of our 


In [34]:
embeddings = embedder.encode(
    paragraphs,
    batch_size=128,
    show_progress_bar=True,
    convert_to_numpy=True,
    normalize_embeddings=True
)

Batches:   0%|          | 0/140 [00:00<?, ?it/s]

In [35]:
normalize_embeddings=True

In [36]:
print(embeddings.shape)

(17793, 384)


In [40]:

dimension = embeddings.shape[1]

index = faiss.IndexFlatIP(dimension)

In [39]:
print(index.ntotal)

0


In [53]:
print(index.ntotal)
print(len(chunks))
print(len(embeddings))

faiss.write_index(
    index,
    "financial_reports.index"
)

17793
17793
17793


In [42]:
metadata = []

for chunk in chunks:
    #print(f"{chunk["metadata"]}")
    tks = chunk["metadata"].split(" ")
    ticker = (tks[1]).split("-")[1]
    section = tks[len(tks)-1].split("-")[1]
    date = tks[len(tks)-2].split("-")[1]
    company = tks[2].split("-")[1]
    metadata.append({
        "ticker": ticker,
        "section": section,
        "year": date,
        "reference": chunk["reference"]
    })

In [43]:
with open("financial_reports_metadata.pkl", "wb") as f:
    pickle.dump(metadata, f)

In [49]:
index = faiss.read_index(
    "financial_reports.index"
)

with open(
    "financial_reports_metadata.pkl",
    "rb"
) as f:

    metadata = pickle.load(f)
index = faiss.IndexFlatIP(384)
index.add(embeddings)

In [54]:
question = "What supplier risks does NVIDIA face in 2025?"

query_embedding = embedder.encode(
    [question],
    convert_to_numpy=True#,
    #normalize_embeddings=True
)

print(query_embedding.shape)
print(query_embedding[0][:10])

(1, 384)
[-0.06139139  0.03560927  0.04088581 -0.01559881  0.03590084 -0.01075188
 -0.05151477  0.00168402 -0.0181034   0.00355046]


In [51]:
scores, ids = index.search(
    query_embedding,
    k=50
)
print(scores)
print(ids)

[[0.70604146 0.6986399  0.6986399  0.69799083 0.69799083 0.69799083
  0.6962707  0.6962707  0.69318354 0.69318354 0.69141185 0.69141185
  0.6907746  0.68690675 0.68690675 0.68642503 0.6840646  0.6840646
  0.68129677 0.68129677 0.6773247  0.6702285  0.6702285  0.66779006
  0.6611482  0.6611482  0.6606936  0.6606936  0.6605904  0.6605904
  0.6598128  0.6598128  0.6598128  0.6587901  0.6587901  0.6587901
  0.6587901  0.6585526  0.6585526  0.6585526  0.65735835 0.65735835
  0.6571115  0.6571114  0.6562192  0.6562192  0.65599453 0.65599453
  0.65566456 0.65566456]]
[[ 1751 15623   732 14612  6308  1468 15769 15029  3554  3463 14216  4363
   6254  9123  8424 15251  7913   879 14892 13474 11039 15799  4700 17402
  10143  4708 13226 12669  8477  3071 17036  9948  3579 17672 11132  9480
   7412 12777  4807  1606 12089  4966  5349 12216  5605  4857  7769   948
   4849  3983]]


In [55]:
for score, idx in zip(scores[0], ids[0]):
    chunk = metadata[idx]

    file_output = open("retrieved_results.jsonl", "a")
    dict = {}
    dict["score"] = f"{score:.4f}"
    dict["ticker"] = chunk["ticker"]
    dict["section"] = chunk["section"]
    dict["year"] = chunk["section"]
    dict["reference"] = chunk["reference"]
    json.dump(dict, file_output)





Score: 0.7060
NVDA
1A
2024
Risk Factors Summary Risks Related to Our Industry and Markets &#8226; Failure to meet the evolving needs of our industry may adversely impact our financial results. &#8226; Competition could adversel
------------------------------------------------------------
Score: 0.6986
NVDA
1A
2026
&#8226; Dependency on third-party suppliers and their technology to manufacture, assemble, test, or package our products reduces our control over product quantity and quality, manufacturing yields, an
------------------------------------------------------------
Score: 0.6986
NVDA
1A
2026
&#8226; Dependency on third-party suppliers and their technology to manufacture, assemble, test, or package our products reduces our control over product quantity and quality, manufacturing yields, an
------------------------------------------------------------
Score: 0.6980
NVDA
1
2021
. These and other risks related to competition are more fully described in the Risk Factors entitled &#8220

In [ ]:
#keep
#Copy data to drive
import os
import shutil
from google.colab import drive

# 1. Mount your Google Drive
#drive.mount('/content/drive')

# 2. Define source and destination folders
# Replace 'my_folder' with the exact folder path where your .json files currently are
source_dir = '.'
# Replace 'My Drive/TargetFolder' with the Drive folder you want to copy to
destination_dir = '/content/drive/MyDrive/capstone_usable_qa_data'

# Create the destination directory if it doesn't exist
os.makedirs(destination_dir, exist_ok=True)

# 3. Find and copy all .json files
for filename in os.listdir(source_dir):
    if filename.endswith('retrieved_results.json'):
        source_file = os.path.join(source_dir, filename)
        destination_file = os.path.join(destination_dir, filename)

        shutil.copy2(source_file, destination_file)
        print(f"Copied: {filename}")

print("All .json files copied successfully!")